# Zero-shot baseline on Colab

Runs the zero-shot baseline (`docs/cloud.md`) with one script. This notebook only drives
`scripts/run_baseline.py` and holds no logic of its own.

**Before you start**
1. *Runtime → Change runtime type → GPU: **L4** or **A100**.* The free T4 has no bf16; don't use it.

Run the cells in order. Results are saved to Google Drive in `MyDrive/voxtral-runs`,
so a disconnect loses nothing: reconnect, run cells 2, 3 and 4 again, and the
script continues where it stopped.

## 1 · Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → L4 or A100"
name = torch.cuda.get_device_name(0)
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {memory_gb:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")
assert torch.cuda.is_bf16_supported() and memory_gb > 20, "Need an L4 or A100, not a T4"

## 2 · Mount Drive, get the code, install

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
url = "https://github.com/HFossdal/voxtral-norwegian.git"
if os.path.exists("/content/voxtral-norwegian"):
    !git -C /content/voxtral-norwegian pull -q {url} main
else:
    !git clone -q {url} /content/voxtral-norwegian
%cd /content/voxtral-norwegian
!git log --oneline -1
!pip install -q -e .
# Colab's preinstalled torchvision is built for its own torch version, not the
# pinned one, and can crash on import. This project doesn't use it.
!pip uninstall -y -q torchvision
# Same for torchao: peft refuses to add LoRA layers when an old torchao is
# installed (Colab ships 0.10; peft 0.21 needs >= 0.16). Not used here either.
!pip uninstall -y -q torchao
# pip then warns about other preinstalled Colab packages (cudf, diffusers, ...)
# wanting other versions of pandas, numba or pyarrow. They are not used here.

## 3 · Restart the runtime

The install replaces Colab's own torch and transformers with the project's pinned
versions. They are only used after a restart. This cell restarts the runtime on
purpose: Colab will report that the session crashed. **That is expected.** Then go on
with cell 4. You don't need to run cells 1–2 again.

The red `ERROR: pip's dependency resolver ...` lines above are warnings about
Colab packages this project never imports. The install itself succeeded if
it printed `Building editable for voxnor ... done`.


In [ ]:
import os

os.kill(os.getpid(), 9)

## 4 · Run the baseline

This is the long step. It runs smoke tests for both models, then four validation runs,
then chooses Mini's language setting, then test for two systems, then the reports.
The log prints progress and speed (× real time). Validation + test are 18.7 h of audio
per system.

In [ ]:
%cd /content/voxtral-norwegian
!python scripts/run_baseline.py --config configs/cloud.yaml --output-root /content/drive/MyDrive/voxtral-runs

## 5 · Look at the results

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

root = Path("/content/drive/MyDrive/voxtral-runs")
display(Markdown((root / "reports" / "baseline_test.md").read_text()))
print((root / "baseline_choice.json").read_text())

Everything worth keeping is also in `MyDrive/voxtral-runs/baseline_results.zip`.
Download it from Drive, or run the cell below.

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/voxtral-runs/baseline_results.zip")